In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install faiss-cpu sentence-transformers transformers

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline

train=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

kb=[]
for idx, row in train.iterrows():
    correct_letter=row['answer']
    kb.append(str(row[correct_letter]))

model=SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings=model.encode(kb, show_progress_bar=False)
index=faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 77.0 MB/s eta 0:00:00


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (4 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])]
ans_150 = str(row_150[row_150['answer']])

result = zs(prompt_150, candidate_labels=labels_150)
score = result['scores'][result['labels'].index(ans_150)]
print("Q1",round(score, 3))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q1 0.384


In [4]:
q_emb=model.encode([prompt_150])
D, I=index.search(q_emb, k=10)
retrieved_indices=I[0].tolist()
rank=retrieved_indices.index(150)+1 if 150 in retrieved_indices else None
print("Q2",rank)

Q2 10


In [5]:
cross_encoder=CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10=[kb[i] for i in retrieved_indices]
pairs=[[prompt_150, doc] for doc in docs_10]
ce_scores=cross_encoder.predict(pairs)
sorted_order=np.argsort(ce_scores)[::-1]
sorted_indices=[retrieved_indices[i] for i in sorted_order]
ce_rank=sorted_indices.index(150)+1
print("Q3",ce_rank)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Q3 1


In [6]:
row_42=train.iloc[42]
prompt_42=str(row_42['prompt'])
q_emb_42=model.encode([prompt_42])
D5, I5=index.search(q_emb_42, k=5)
docs_5=[kb[i] for i in I5[0]]
concatenated=" ".join(docs_5)
rag_string=f"Context: {concatenated} Question: {prompt_42}"

tokenizer=AutoTokenizer.from_pretrained("bert-base-uncased")
tokens=tokenizer(rag_string, truncation=False)
print("Q4",len(tokens['input_ids']))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q4 216


In [7]:
true_doc_150=kb[150]
rag_string_150=f"Context: {true_doc_150} Question: {prompt_150}"
result5=zs(rag_string_150, candidate_labels=labels_150)
score5=result5['scores'][result5['labels'].index(ans_150)]
print("Q5",round(score5, 3))

Q5 0.989


In [8]:
adversarial_doc=kb[999]
adv_string=f"Context: {adversarial_doc} Question: {prompt_150}"
result6=zs(adv_string, candidate_labels=labels_150)
score6=result6['scores'][result6['labels'].index(ans_150)]
print("Q6",round(score6, 3))

Q6 0.529


In [9]:
hits=0
for idx in range(100):
    row=train.iloc[idx]
    prompt=str(row['prompt'])
    correct_text=str(row[row['answer']])
    q_emb=model.encode([prompt])
    D, I=index.search(q_emb, k=5)
    retrieved_docs=[kb[i] for i in I[0]]
    if any(correct_text in doc for doc in retrieved_docs):
        hits+=1

hit_rate=round(hits/100*100, 1)
print("Q7",hit_rate)

Q7 73.0


In [10]:
def map_at_3(ranked_letters, correct_letter):
    if correct_letter in ranked_letters:
        rank=ranked_letters.index(correct_letter)+1
        return 1.0/rank
    return 0.0

scores=[]
for idx in range(20):
    row=train.iloc[idx]
    prompt=str(row['prompt'])
    letters=['A', 'B', 'C', 'D', 'E']
    option_labels=[str(row[l]) for l in letters]
    correct_letter=row['answer']

    q_emb=model.encode([prompt])
    D, I=index.search(q_emb, k=5)
    docs_5=[kb[i] for i in I[0]]

    pairs=[[prompt, doc] for doc in docs_5]
    ce_scores=cross_encoder.predict(pairs)
    best_doc=docs_5[np.argmax(ce_scores)]

    rag_string=f"Context: {best_doc} Question: {prompt}"
    result=zs(rag_string, candidate_labels=option_labels)

    label_to_letter={str(row[l]): l for l in letters}
    ranked_letters=[label_to_letter[lbl] for lbl in result['labels']]
    top3=ranked_letters[:3]

    scores.append(map_at_3(top3, correct_letter))

final_map3=round(np.mean(scores), 3)
print("Q8",final_map3)

Q8 0.975
